## **Claim Check AI**
**Input:** One written Claim

**Evidence source:** a small collection of science facts.

**output:** Support, refuted, or insufficient evidence

In [ ]:
!pip install numpy
!pip install pandas
!pip install streamlit
!pip install joblib
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 29.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
import pandas as pd
import numpy as np
import streamlit as st
import joblib
import tqdm

Dataset sample 1000

In [ ]:
df = pd.read_csv("ClaimCheckAI_practice_1000.csv")

In [ ]:
print(df.shape)
df.columns
display(df.head())

(1000, 7)


,claim,label,evidence,source,source_url,example_family,data_origin
0,"For this example, consider: the claim is that ...",SUPPORTS,HTML is a markup language used to structure we...,MDN Web Docs,https://developer.mozilla.org/en-US/docs/Web/HTML,fact_19,synthetic_template_expanded
1,Fact-check the following: one claim says that ...,REFUTES,The human heart has four chambers.,Cleveland Clinic,https://my.clevelandclinic.org/,fact_25,synthetic_template_expanded
2,"Claim for review: in simple terms, water is ma...",SUPPORTS,Water is made of hydrogen and oxygen.,USGS,https://www.usgs.gov/special-topics/water-scie...,fact_7,synthetic_template_expanded
3,"For this example, consider: the Sun is a planet.",REFUTES,The Sun is a star.,NASA,https://science.nasa.gov/solar-system/sun/,fact_4,synthetic_template_expanded
4,"In simple terms, hTTP is a type of graphics card.",REFUTES,HTTP is a protocol used for web communication.,MDN Web Docs,https://developer.mozilla.org/en-US/docs/Web/HTTP,fact_22,synthetic_template


In [ ]:
df.isnull().sum()

,0
claim,0
label,0
evidence,0
source,0
source_url,0
example_family,0
data_origin,0


In [ ]:
df[df["source_url"].isnull()]

,claim,label,evidence,source,source_url,example_family,data_origin


In [ ]:
print(df["label"].value_counts())

label
SUPPORTS           487
REFUTES            469
NOT ENOUGH INFO     44
Name: count, dtype: int64


change the lebel into numeric so the machine learning model can understand and word on it

In [ ]:
label_map = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT ENOUGH INFO": 2
}
df["label_id"] = df["label"].map(label_map)
display(df[["claim", "label", "label_id"]])

,claim,label,label_id
0,"For this example, consider: the claim is that ...",SUPPORTS,0
1,Fact-check the following: one claim says that ...,REFUTES,1
2,"Claim for review: in simple terms, water is ma...",SUPPORTS,0
3,"For this example, consider: the Sun is a planet.",REFUTES,1
4,"In simple terms, hTTP is a type of graphics card.",REFUTES,1
...,...,...,...
995,"In simple terms, the Sun is a planet.",REFUTES,1
996,"In simple terms, python is a web browser.",REFUTES,1
997,Claim for review: one claim says that pure wat...,REFUTES,1
998,"According to this statement, neptune is the cl...",REFUTES,1


In [ ]:
df["combine_text"] = df["claim"]+ " " + df["evidence"]
display(df[["combine_text", "label"]].head())

,combine_text,label
0,"For this example, consider: the claim is that ...",SUPPORTS
1,Fact-check the following: one claim says that ...,REFUTES
2,"Claim for review: in simple terms, water is ma...",SUPPORTS
3,"For this example, consider: the Sun is a plane...",REFUTES
4,"In simple terms, hTTP is a type of graphics ca...",REFUTES


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["combine_text"])

print("TF-IDF matrix shape", X.shape)

TF-IDF matrix shape (1000, 224)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def search_evidence(query, top_k=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, X).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = df.iloc[top_indices][
        ["claim", "evidence", "source_url"]
    ].copy()

    results["similarity"] = scores[top_indices]
    return results
display(search_evidence("jupyter is the largest planet", top_k=3))

,claim,evidence,source_url,similarity
427,The claim is that jupiter is the largest plane...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.578135
722,The statement to verify is: the claim is that ...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.577499
688,Jupiter is the largest planet in the Solar Sys...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.566022


Create semantic embeddings for the device

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

evidence_embeddings = model.encode(
    df["evidence"].tolist(),
    show_progress_bar=True
)
print("Embeddings shape:", evidence_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings shape: (1000, 384)


search evidense by meaning

In [ ]:
from sentence_transformers import util

def semantic_search(query, top_k=3):
  query_embedding = model.encode(query, convert_to_tensor=True)

  scores = util.cos_sim(
      query_embedding,
      evidence_embeddings
  )[0]
  top_indices = scores.argsort(descending=True)[:top_k]

  results = df.iloc[top_indices][
      ["claim", "evidence", "source_url"]
  ].copy()

  results["similarity"] = scores[top_indices]
  return results

display(semantic_search("which planet is the largest", top_k=3))



,claim,evidence,source_url,similarity
210,One claim says that jupiter is smaller than Ea...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.800061
54,It is true that jupiter is smaller than Earth.,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.800061
625,The statement to verify is: according to this ...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.800061


In [ ]:
!pip install transformers torch

Now load the pretrained NLI model

In [ ]:
from transformers import pipeline

nli_model = pipeline(
    "text-classification",
    model= "cross-encoder/nli-deberta-v3-small"
)
print("NLI model loaded successfully!")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  568MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

NLI model loaded successfully!


test the model

In [ ]:
result = nli_model({
    "text": "Jupiter is the largest planet in the Solar System",
    "text_pair": "Earth is the largest planet in the Solar System"
})
print(result)

{'label': 'contradiction', 'score': 0.9991140961647034}


In [ ]:
def check_claim(claim, top_k=3):
  evidence_results = semantic_search(claim, top_k=top_k)
  for _, row in evidence_results.iterrows():
    result = nli_model({
        "text": row["evidence"],
        "text_pair": claim
    })
    print("claim:", claim)
    print("Evidence:", row["evidence"])
    print("Source:", row["source_url"])
    print("NLI result:", result)
    print("-" * 40)

check_claim("Earth is the largest planet in the Solar System.")

claim: Earth is the largest planet in the Solar System.
Evidence: Jupiter is the largest planet in the Solar System.
Source: https://science.nasa.gov/solar-system/planets/jupiter/
NLI result: {'label': 'contradiction', 'score': 0.9994789958000183}
----------------------------------------
claim: Earth is the largest planet in the Solar System.
Evidence: Jupiter is the largest planet in the Solar System.
Source: https://science.nasa.gov/solar-system/planets/jupiter/
NLI result: {'label': 'contradiction', 'score': 0.9994789958000183}
----------------------------------------
claim: Earth is the largest planet in the Solar System.
Evidence: Jupiter is the largest planet in the Solar System.
Source: https://science.nasa.gov/solar-system/planets/jupiter/
NLI result: {'label': 'contradiction', 'score': 0.9994789958000183}
----------------------------------------


This will make the output more easier

In [ ]:
def check_claim(claim, top_k=3):
  results = semantic_search(claim, top_k=top_k)
  for _, row in results.iterrows():
    prediction = nli_model({
        "text": row["evidence"],
        "text_pair": claim
    })

    print("Evidence:", row["evidence"])
    print("Verdict:", prediction["label"].upper())
    print("Source:", row["source_url"])
    print("Confidence:", round(prediction["score"], 4))
    print("-" * 40)

check_claim("Earth is the largest planet in the Solar System.")

Evidence: Jupiter is the largest planet in the Solar System.
Verdict: CONTRADICTION
Source: https://science.nasa.gov/solar-system/planets/jupiter/
Confidence: 0.9995
----------------------------------------
Evidence: Jupiter is the largest planet in the Solar System.
Verdict: CONTRADICTION
Source: https://science.nasa.gov/solar-system/planets/jupiter/
Confidence: 0.9995
----------------------------------------
Evidence: Jupiter is the largest planet in the Solar System.
Verdict: CONTRADICTION
Source: https://science.nasa.gov/solar-system/planets/jupiter/
Confidence: 0.9995
----------------------------------------


Avoid showing the same evidence multiple times

In [ ]:
df_unique = df.drop_duplicates(subset=["evidence"]).reset_index(drop=True)

evidence_embeddings = model.encode(
    df_unique["evidence"].tolist(),
    convert_to_tensor = True
)

print("Original rows:", len(df))
print("Unique evidence passages:", len(df_unique))

Original rows: 1000
Unique evidence passages: 35


Now new sementic search on df_unique

In [ ]:
def semantic_search(query, top_k=3):
  query_embedding = model.encode(query, convert_to_tensor=True)

  scores = util.cos_sim(
      query_embedding, evidence_embeddings
  )[0]

  top_indices = scores.argsort(descending=True)[:top_k].tolist()

  results = df_unique.iloc[top_indices][
      ["claim", "evidence", "source_url"]
  ].copy()

  results["similarity"] = [
      scores[i].item() for i in top_indices
  ]

  return results

display(semantic_search("Which planet is the largest?", top_k=3))

,claim,evidence,source_url,similarity
0,Jupiter is the largest planet in the Solar Sys...,Jupiter is the largest planet in the Solar Sys...,https://science.nasa.gov/solar-system/planets/...,0.787764
2,Earth is a planet in the Solar System.,Earth is the third planet from the Sun.,https://science.nasa.gov/solar-system/planets/...,0.539361
1,The Sun orbits Earth.,Earth orbits the Sun.,https://science.nasa.gov/solar-system/planets/...,0.379346


Updated check_claim() to use the cleaned dataset df_unique

In [ ]:
def check_claim(claim, top_k=3):
    results = semantic_search(claim, top_k=top_k)

    for _, row in results.iterrows():
        prediction = nli_model({
            "text": row["evidence"],
            "text_pair": claim
        })

        print("Claim:", claim)
        print("Evidence:", row["evidence"])
        print("Verdict:", prediction["label"].upper())
        print("Confidence:", round(prediction["score"], 4))
        print("Source:", row["source_url"])
        print("-" * 40)

check_claim("Earth is the largest planet in the Solar System.")

Claim: Earth is the largest planet in the Solar System.
Evidence: Jupiter is the largest planet in the Solar System.
Verdict: CONTRADICTION
Confidence: 0.9995
Source: https://science.nasa.gov/solar-system/planets/jupiter/
----------------------------------------
Claim: Earth is the largest planet in the Solar System.
Evidence: Earth is the third planet from the Sun.
Verdict: NEUTRAL
Confidence: 0.9991
Source: https://science.nasa.gov/solar-system/planets/earth/
----------------------------------------
Claim: Earth is the largest planet in the Solar System.
Evidence: Jupiter is the largest planet in the Solar System.
Verdict: CONTRADICTION
Confidence: 0.9995
Source: https://science.nasa.gov/solar-system/planets/jupiter/
----------------------------------------


The system will produce one final verdict instead of printing three separate predictions

In [ ]:
def verdict(claim, top_k=3):
    results = semantic_search(claim, top_k=top_k)

    for _, row in results.iterrows():
        prediction = nli_model({
            "text": row["evidence"],
            "text_pair": claim
        })

        if prediction["label"].lower() == "contradiction":
            print("Final verdict: REFUTED")
            print("Evidence:", row["evidence"])
            print("Source:", row["source_url"])
            return

    print("verdict: NOT ENOUGH EVIDENCE")

verdict("Jupiter is the largest planet in the Solar System.")

verdict: NOT ENOUGH EVIDENCE


In [ ]:
def final_verdict(claim, top_k=3, threshold=0.5):
    results = semantic_search(claim, top_k=top_k)

    for _, row in results.iterrows():

        # Ignore evidence that is not sufficiently relevant
        if row["similarity"] < threshold:
            continue

        prediction = nli_model({
            "text": row["evidence"],
            "text_pair": claim
        })

        label = prediction["label"].lower()

        if label == "entailment":
            print("Final verdict: SUPPORTED")
            print("Evidence:", row["evidence"])
            print("Source:", row["source_url"])
            return

        elif label == "contradiction":
            print("Final verdict: REFUTED")
            print("Evidence:", row["evidence"])
            print("Source:", row["source_url"])
            return

    print("Final verdict: NOT ENOUGH EVIDENCE")

In [ ]:
final_verdict("Jupiter is the largest planet in the Solar System.")

Final verdict: SUPPORTED
Evidence: Jupiter is the largest planet in the Solar System.
Source: https://science.nasa.gov/solar-system/planets/jupiter/


In [ ]:
final_verdict("Aliens definitely live on a distant exoplanet.")

Final verdict: REFUTED
Evidence: Pure water freezes at about 0 degrees Celsius at standard pressure.
Source: https://www.nist.gov/


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())


Dataset shape: (1000, 9)

Label distribution:
label
SUPPORTS           487
REFUTES            469
NOT ENOUGH INFO     44
Name: count, dtype: int64
